### Imports

In [1]:
import json
from pathlib import Path

import optuna
import torch
from torch import nn

from src.datasets.spike_dataset import SpikeDataset
from src.engine import benchmark_snn, train_one_epoch_snn, validate_snn, get_split_dataloaders
from src.models.snn_1d_encoded_classifier import SNN1DEncodedClassifier
from src.utils import plot_training_history, run_sweep

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Constants

In [2]:
MODEL_NAME = SNN1DEncodedClassifier.NAME

INPUT_DIR = Path('../../processed/audioMNIST')
HYPERPARAMETERS_PATH = Path(f'../../hyperparameters/{MODEL_NAME}.json')
MODEL_PATH = Path(f'../../models/{MODEL_NAME}.pth')

NUM_EPOCHS = 20

### Device

In [3]:
device = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps' if torch.backends.mps.is_available() else
    'cpu'
)
print(f'Using device: {device}')

Using device: mps


### Hyperparameter Tuning

In [ ]:
def objective(trial) -> float:
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    beta = trial.suggest_float('beta_init', 0.5, 0.99)
    slope = trial.suggest_int('slope', 10, 50)

    dataset = SpikeDataset(INPUT_DIR)
    train_dataloader, val_dataloader, _ = get_split_dataloaders(dataset)

    model = SNN1DEncodedClassifier(beta_init=beta, slope=slope).to(device)
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    val_acc = 0.0
    for epoch in range(3):
        train_one_epoch_snn(device, model, criterion, optimiser, train_dataloader, leave=False)
        val_loss, val_acc = validate_snn(device, model, criterion, val_dataloader, leave=False)

        trial.report(val_acc, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return val_acc

run_sweep(objective, HYPERPARAMETERS_PATH, 20)

[I 2026-03-08 21:12:52,239] A new study created in memory with name: no-name-65ef760d-426b-4252-9d95-d6c9fedb151c


Running hyperparameter sweep...


Training:   0%|          | 0/375 [00:00<?, ?batches/s]/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
[I 2026-03-08 21:15:39,980] Trial 0 finished with value: 73.4 and parameters: {'lr': 0.0007950555735859382, 'beta_init': 0.7195545543633364, 'slope': 36}. Best is trial 0 with value: 73.4.
[I 2026-03-08 21:18:21,309] Trial 1 finished with value: 44.13333333333333 and parameters: {'lr': 0.0016944961235999213, 'beta_init': 0.9001368372570078, 'slope': 48}. Best is trial 0 with value: 73.4.
[I 2026-03-08 21:21:05,564] Trial 2 finished with value: 17.2 and parameters: {'lr': 0.0032494402162435558, 'beta_init': 0.9282134903412076, 'slope': 39}. Best is trial 0 with value: 73.4.
[I 2026-03-08 21:23:44,374] Trial 3 finished with value: 74.4 and parameters: {'lr': 0.0074645392320198

### Training

In [ ]:
if __name__ == '__main__':
    dataset = SpikeDataset(INPUT_DIR)
    train_dataloader, val_dataloader, test_dataloader = get_split_dataloaders(dataset)

    # Get one batch from the training loader and make sure it looks good
    features, labels = next(iter(train_dataloader))
    print(f'Features shape: {features.shape}')
    print(f'Labels shape: {labels.shape}')
    print()

    # Load hyperparameters
    hyperparameters = json.load(open(HYPERPARAMETERS_PATH, 'r'))
    print(f'Hyperparameters used: {hyperparameters}')
    print()

    model = SNN1DEncodedClassifier(beta_init=hyperparameters['beta_init'], slope=hyperparameters['slope']).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=hyperparameters['lr'])
    criterion = nn.CrossEntropyLoss()

    print("Training SNN...")
    print(f'Training {model.NAME}...')
    best_acc = 0.0
    train_losses, train_accs = [], []
    val_losses, val_accs = [], []
    for epoch in range(NUM_EPOCHS):
        print(f'[Epoch {epoch + 1}/{NUM_EPOCHS}]')
        train_loss, train_acc = train_one_epoch_snn(device, model, criterion, optimizer, train_dataloader)
        train_losses.append(train_loss)
        train_accs.append(train_acc)

        val_loss, val_acc = validate_snn(device, model, criterion, val_dataloader)
        val_losses.append(val_loss)
        val_accs.append(val_acc)

        if val_acc > best_acc:
            best_accuracy = val_acc
            torch.save(model.state_dict(), MODEL_PATH)

        print(f'Train Loss: {train_loss:.2f} | Train Accuracy: {train_acc:.2f}% | Val Loss: {val_loss:.2f} | Val Accuracy: {val_acc:.2f}%')
        print()

    print(f'Best model had an accuracy of {best_acc:.2f}%.')
    print(f'Running final test...')
    checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=True)
    model.load_state_dict(checkpoint, strict=True)
    model.to(device)

    test_accuracy, avg_acs_per_inference = benchmark_snn(device, model, test_dataloader)
    print(f'Test accuracy: {test_accuracy:.2f}% | Total ACs: {avg_acs_per_inference:.0f}')

    plot_training_history(train_losses, train_accs, val_losses, val_accs)